<a id="encrypted-machine-learning-activations"></a>
# Encrypted Machine Learning: Activations

This tutorial covers `src/concrete_fhe_toolkit/ml/activations.py`. Machine learning models require non-linear activation functions to learn complex patterns. This module provides FHE-compatible versions of popular activations like ReLU, Leaky ReLU, Unit Step, and Softmax.

<a id="relu-leaky-relu"></a>
## ReLU & Leaky ReLU

In [ ]:
from concrete import fhe
from concrete_fhe_toolkit.ml.activations import relu, leaky_relu

def test_relus(val: int):
    return relu(val), leaky_relu(val, alpha=0.1)

compiler = fhe.Compiler(test_relus, {"val": "encrypted"})
inputset = [(10,), (-10,), (0,)]
circuit = compiler.compile(inputset)

# Positive value
assert circuit.encrypt_run_decrypt(10) == (10, 10)

# Negative value: relu is 0, leaky_relu with alpha=0.1 of -10 is -1
assert circuit.encrypt_run_decrypt(-10) == (0, -1)
print("✅ Encrypted ReLU and Leaky ReLU passed!")

<a id="unit-step-threshold"></a>
## Unit Step & Threshold

In [ ]:
from concrete_fhe_toolkit.ml.activations import unit_step, threshold_activation

def test_steps(val: int, thresh: int):
    return unit_step(val), threshold_activation(val, thresh)

compiler = fhe.Compiler(test_steps, {"val": "encrypted", "thresh": "encrypted"})
inputset = [(5, 10), (-3, 0), (10, 5)]
circuit = compiler.compile(inputset)

# 5 is >= 0 (unit_step=1), but NOT >= 10 (threshold=0)
assert circuit.encrypt_run_decrypt(5, 10) == (1, 0)

# -3 is NOT >= 0 (unit_step=0)
assert circuit.encrypt_run_decrypt(-3, 0) == (0, 0)
print("✅ Encrypted Unit Step and Threshold passed!")

<a id="softmax"></a>
## Softmax

In [ ]:
from concrete_fhe_toolkit.ml.activations import make_softmax

softmax_fn = make_softmax(min_input=-50, max_input=50, input_scale=10, probability_scale=100)

def test_softmax(s1: int, s2: int, s3: int):
    # Softmax requires a list of inputs
    return softmax_fn([s1, s2, s3])

compiler = fhe.Compiler(test_softmax, {"s1": "encrypted", "s2": "encrypted", "s3": "encrypted"})
inputset = [(0, 0, 0), (20, -10, 5)]
circuit = compiler.compile(inputset)

# All scores equal -> ~33% probability for each (scaled by 100)
probs = circuit.encrypt_run_decrypt(0, 0, 0)
assert sum(probs) >= 99  # close to 100%
assert probs[0] == 33
print("✅ Encrypted Softmax passed!")